# FOV-Condition Monitor Agent — standalone demo

Uses `FOVConditionMonitorAgent` on its own (no BO, no composition) to gate the
start of an experiment on a measured ERK baseline, rather than a guessed wait.

When cells are moved to the microscope, mechanical-stress ERK pushes CNR up
to ~ 1.5–2; in starvation medium it decays over hours. Instead of waiting a
fixed 4 h and hoping, the monitor:

1. Picks `N_MONITOR_FOVS` candidate positions inside one **sacrificial** well
   via a `FOVFinderAgent` (built internally — same machinery as the FOV
   finder demos in [32_fov_finder/](../32_fov_finder/)).
2. Images them with the user's imaging channels, segments, and runs
   `FE_ErkKtr` to get per-cell CNR.
3. Pools per-cell CNR across all monitored FOVs and tests it against a
   `FOVCondition` (e.g. *75 % of cells have CNR < 1.0*).
4. Sleeps `check_interval_s` and repeats — until the condition fires (and
   the agent returns `triggered=True`) or `timeout_s` elapses.

The sacrificial well is consumed by the monitor; the real experiment must
then run on a fresh well. The agent records a per-round trajectory parquet
that you can plot to show the CNR decay curve and the trigger point.

See [`simple_fov_finder_demo.ipynb`](../32_fov_finder/simple_fov_finder_demo.ipynb)
for the FOV-finder building block, and
[`bo_erk_oscillation_testv13_auc_fixed_budget.ipynb`](../31_bo_optimisation/bo_erk_oscillation_testv13_auc_fixed_budget.ipynb)
for the full BO that the monitor gates.

## 1. Microscope and channel setup

The monitor needs miRFP (segmentation, nuclear marker) **and** mScarlet3
(ERK-KTR signal) — `FE_ErkKtr` reads channel index 1 to compute CNR, so
both channels must be acquired.

In [1]:
from faro.microscope.pertzlab.jungfrau import Jungfrau
from faro.core.data_structures import PowerChannel

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

# miRFP at index 0 (segmented), mScarlet3 at index 1 (the ERK-KTR channel
# FE_ErkKtr reads to compute per-cell CNR). The cyan stim channel is
# intentionally NOT used here — it would pre-activate CRY2.
imaging_channels = (
    PowerChannel(config="miRFP", exposure=125, group="TTL_ERK", power=95),
    PowerChannel(config="mScarlet3", exposure=125, group="TTL_ERK", power=95),
)

In [2]:
mic.mmc.setProperty(
    "TIPFSStatus", "State", "On"
)  # Focus already done -- turn on PFS so the monitor doesn't drive Z.
# If not done, focus through the napari-micromanager GUI first:

from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)

## 2. Plate calibration

Same `WellPlatePlan` JSON saved by the `pymmcore-widgets` MDA plate widget
that the FOV-finder demos use.

In [ ]:
PLATE_CALIBRATION_PATH = "./calib_plate_96.json"

# Quick sanity check
from useq import WellPlatePlan

plan = WellPlatePlan.from_file(PLATE_CALIBRATION_PATH)
print(f"plate: {plan.plate.name}  rows={plan.plate.rows}  cols={plan.plate.columns}")
print(f"well_size mm: {plan.plate.well_size}  circular: {plan.plate.circular_wells}")

## 3. Segmentator and feature extractor

The monitor needs:

- a `Segmentator` (here `CellposeV4` on miRFP) that gives a nuclear mask, and
- a `FeatureExtractor` (here `FE_ErkKtr`) that turns the mask + image stack
  into a per-cell DataFrame with a `cnr` column.

In [ ]:
from faro.segmentation.cellpose_v4 import CellposeV4
from faro.feature_extraction.erk_ktr import FE_ErkKtr

segmentator = CellposeV4()
feature_extractor = FE_ErkKtr("labels")

## 4. Configure the monitor's scout FOV finder

A `FOVFinderAgent` pointed at a single **sacrificial** well, used by the
monitor each round to pick fresh candidate FOVs and image them.

Three settings are non-negotiable for monitoring; the agent will raise if
they're wrong:

- `wells=[SACRIFICIAL_WELL]` and `wells_per_phase=1` — one well only.
- `cycle_wells=True` — the well queue refills, so the same well can be
  re-imaged every monitoring round.
- `fov_conditions=[...]` with a `feature_extractor` — required so per-cell
  CNR is extracted into `last_run["fov_features"]` (which the monitor pools
  across FOVs). Pass the same `FOVCondition` you give the monitor.

`fovs_per_well` is the number of FOVs imaged per round (the monitor's
"how many FOVs do you want to check" knob).

In [ ]:
from faro.agents import FOVCondition, FOVFinderAgent

SACRIFICIAL_WELL = "B2"  # this well will be consumed by the monitor
N_MONITOR_FOVS = 4  # how many FOVs to image each round

# Readiness condition: CNR < 1.0 in >= 75 % of monitored cells.
# Same threshold the BO's FOV-finder pre-screen uses, so "monitor triggers"
# means "the BO's FOV finder will now succeed on fresh wells".
ready = FOVCondition("cnr", "below", 1.0, min_fraction=0.75)

monitor_finder = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=[SACRIFICIAL_WELL],
    wells_per_phase=1,
    fovs_per_well=N_MONITOR_FOVS,
    n_candidates_per_well=N_MONITOR_FOVS,
    border_um=1000.0,
    min_distance_um=750.0,
    min_cells=30,
    imaging_channels=imaging_channels,  # miRFP (seg) + mScarlet3 (CNR)
    segmentator=segmentator,
    seg_channel_index=0,  # segment miRFP
    feature_extractor=feature_extractor,
    fov_conditions=[ready],  # enables per-cell CNR extraction
    cycle_wells=True,  # REQUIRED: re-image the well each round
    strict_count=False,
    z=None,  # leave Z untouched (PFS holds focus)
    verbose=False,
)

## 5. Configure the starvation monitor

Wraps the scout finder above with the timing loop and trigger logic.

Key knobs:

- `condition` — the readiness check. Re-use `ready` from above so the
  trigger and the FOV-finder pre-screen are the same.
- `check_interval_s` — seconds between rounds. Production: `20 * 60` (20 min).
- `timeout_s` — total time before giving up. Production: `20 * 60 * 60` (20 h).
- `storage_path` — if set, writes `condition_monitor.parquet` (round,
  elapsed_min, n_cells_pooled, frac_below, median_cnr, triggered).

In [ ]:
import os
from faro.agents import FOVConditionMonitorAgent

# --- For a dry-run / demo: short interval + short timeout so the loop
# cycles a few times in a few minutes.  Use the production values commented
# below for a real overnight run.
CHECK_INTERVAL_S = 60  # production: 20 * 60   = 20 min
TIMEOUT_S = 5 * 60  # production: 20 * 60 * 60 = 20 h

STORAGE_PATH = os.path.abspath("./fov_condition_monitor_demo")
os.makedirs(STORAGE_PATH, exist_ok=True)

monitor = FOVConditionMonitorAgent(
    fov_finder=monitor_finder,
    fov_conditions=[ready],
    check_interval_s=CHECK_INTERVAL_S,
    timeout_s=TIMEOUT_S,
    storage_path=STORAGE_PATH,
)
print(
    f"monitor armed: trigger when {ready.feature} {ready.operator} {ready.threshold:g} "
    f"in >= {ready.min_fraction * 100:.0f}% of cells"
)
print(f"check every {CHECK_INTERVAL_S / 60:.1f} min, timeout {TIMEOUT_S / 60:.0f} min")
print(f"trajectory parquet -> {STORAGE_PATH}/condition_monitor.parquet")

## 6. Run the monitor

`monitor.run()` is **blocking** — the cell sits in the loop until the
trigger fires or the timeout elapses. Each round prints a one-line summary
(pooled cell count, median CNR, fraction-below, target). On trigger it
prints `TRIGGERED after ... min`; on timeout it prints `TIMEOUT` and
returns `triggered=False`.

In [ ]:
result = monitor.run()
print()
print(f"triggered     : {result['triggered']}")
print(f"n_rounds      : {result['n_rounds']}")
print(f"elapsed (min) : {result['elapsed_min']:.1f}")
# (final per-condition fractions are in the trajectory's last row)

## 7. Plot the decline curve

The agent saves a per-round trajectory to
`<storage_path>/condition_monitor.parquet`. Plotting `elapsed_min` against
`median_cnr` and `frac_below` shows the cells starving down over time and
marks where the trigger fired.

This is the "intelligent vs guessed start" figure: overlay your previous
fixed-wait time (e.g. 240 min) as a vertical line on the same plot to see
the gap between the guess and the data-driven trigger.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_traj = pd.read_parquet(os.path.join(STORAGE_PATH, "condition_monitor.parquet"))
df_traj

In [ ]:
GUESSED_WAIT_MIN = 240  # the fixed 4 h wait the monitor replaces

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

ax = axes[0]
ax.plot(df_traj["elapsed_min"], df_traj["c0_median"], "o-", color="C0")
ax.axhline(
    ready.threshold,
    ls="--",
    color="grey",
    label=f"threshold {ready.feature}={ready.threshold:g}",
)
ax.axvline(
    GUESSED_WAIT_MIN, ls=":", color="C3", label=f"guessed wait ({GUESSED_WAIT_MIN} min)"
)
if result["triggered"]:
    ax.axvline(
        result["elapsed_min"],
        ls="-",
        color="C2",
        label=f"agent triggered ({result['elapsed_min']:.0f} min)",
    )
ax.set_xlabel("elapsed (min)")
ax.set_ylabel(f"median {ready.feature}")
ax.set_title("CNR decay over time")
ax.legend(loc="best", fontsize=8)

ax = axes[1]
ax.plot(df_traj["elapsed_min"], df_traj["c0_frac"] * 100, "o-", color="C4")
ax.axhline(
    ready.min_fraction * 100,
    ls="--",
    color="grey",
    label=f"target {ready.min_fraction * 100:.0f}%",
)
ax.axvline(GUESSED_WAIT_MIN, ls=":", color="C3", label="guessed wait")
if result["triggered"]:
    ax.axvline(result["elapsed_min"], ls="-", color="C2", label="agent triggered")
ax.set_xlabel("elapsed (min)")
ax.set_ylabel(f"% cells {ready.operator} {ready.threshold:g}")
ax.set_title("Fraction below threshold over time")
ax.legend(loc="best", fontsize=8)

fig.suptitle(f"Starvation monitor — well {SACRIFICIAL_WELL}", fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Wiring into an experiment

Once you trust the monitor offline, drop it into the v13 BO notebook as a
phase-0 gate:

```python
# Reserve one extra well as the sacrificial one
SACRIFICIAL_WELL = WELLS[0]
BO_WELLS = WELLS[1:]               # the BO experiment runs on these
fov_finder = FOVFinderAgent(..., wells=BO_WELLS, ...)
...
monitor = FOVConditionMonitorAgent(
    fov_finder=monitor_finder,     # pointed at [SACRIFICIAL_WELL]
    fov_conditions=[ready],
    check_interval_s=20 * 60,
    timeout_s=20 * 60 * 60,
    storage_path=path,
)

# In the run cell, BEFORE the composed BO loop:
result = monitor.run()             # blocks until ready / timeout
if not result["triggered"]:
    print("WARNING: starvation timeout — inspect trajectory before trusting the run")
composed_agent.run()
```

See [`bo_erk_oscillation_testv13_auc_fixed_budget.ipynb`](../31_bo_optimisation/bo_erk_oscillation_testv13_auc_fixed_budget.ipynb)
for the full integration.

## 9. Without a FOV finder — bench monitoring

`fov_finder` is polymorphic. So far we passed a `FOVFinderAgent` for
well-based scouting. The same constructor also accepts:

- a single `FovPosition` — image that one spot every round;
- a `list[FovPosition]` — image those spots every round;
- `None` (default) — image at the **current** stage position, fetched
  once at `run()` start via `mic.get_position()`.

For these three forms the agent has no finder to lift the imaging stack
from, so you pass `microscope` + `imaging_channels` + `segmentator` +
`feature_extractor` directly (and optionally `seg_channel_index`). These
args are *ignored* when `fov_finder` is a `FOVFinderAgent` — that already
carries them.

Below: monitor at the **current** stage position (navigate the
microscope to where you want first, *then* run the cell). Same dry-run
defaults as Section 5 so the loop cycles a few times in ~5 min.

In [ ]:
from faro.agents import FOVConditionMonitorAgent

bench_monitor = FOVConditionMonitorAgent(
    # fov_finder=None (default) -> use the current stage position
    microscope=mic,
    imaging_channels=imaging_channels,  # miRFP + mScarlet3
    segmentator=segmentator,
    feature_extractor=feature_extractor,
    seg_channel_index=0,
    fov_conditions=[ready],
    check_interval_s=CHECK_INTERVAL_S,  # short dry-run interval from §5
    timeout_s=TIMEOUT_S,
    storage_path=os.path.join(STORAGE_PATH, "bench_current"),
)
bench_result = bench_monitor.run()
print()
print(f"triggered     : {bench_result['triggered']}")
print(f"n_rounds      : {bench_result['n_rounds']}")
print(f"elapsed (min) : {bench_result['elapsed_min']:.1f}")

### Other forms — explicit positions

A single `FovPosition` images one spot every round; a list of them images
each in turn (re-using the same imaging stack). These are useful when
you've manually identified the spots you want to monitor (e.g. from a
napari snapshot) and don't want to bother building a `WellPlatePlan`.

The cells below only **construct** the agents to show the API; uncomment
`.run()` to actually run them.

In [ ]:
from faro.core.utils import FovPosition

# --- One explicit position ---
one_spot = FovPosition(x=45000.0, y=-23000.0, z=None, name="my_spot")

monitor_one = FOVConditionMonitorAgent(
    fov_finder=one_spot,  # <-- single FovPosition
    microscope=mic,
    imaging_channels=imaging_channels,
    segmentator=segmentator,
    feature_extractor=feature_extractor,
    seg_channel_index=0,
    fov_conditions=[ready],
    check_interval_s=CHECK_INTERVAL_S,
    timeout_s=TIMEOUT_S,
    storage_path=os.path.join(STORAGE_PATH, "bench_one"),
)
print(f"monitor_one armed in mode={monitor_one.mode!r}")
# monitor_one.run()    # uncomment to actually run

# --- A list of positions ---
spots = [
    FovPosition(x=45000.0, y=-23000.0, z=None, name="A"),
    FovPosition(x=36000.0, y=-22500.0, z=None, name="B"),
    FovPosition(x=28000.0, y=-23500.0, z=None, name="C"),
]

monitor_list = FOVConditionMonitorAgent(
    fov_finder=spots,  # <-- list[FovPosition]
    microscope=mic,
    imaging_channels=imaging_channels,
    segmentator=segmentator,
    feature_extractor=feature_extractor,
    seg_channel_index=0,
    fov_conditions=[ready],
    check_interval_s=CHECK_INTERVAL_S,
    timeout_s=TIMEOUT_S,
    storage_path=os.path.join(STORAGE_PATH, "bench_list"),
)
print(f"monitor_list armed in mode={monitor_list.mode!r}  ({len(spots)} positions)")
# monitor_list.run()   # uncomment to actually run